In [ ]:
# STEP 1: INSTALL AND IMPORT REQUIRED LIBRARIES
# ============================================================================

!pip install beautifulsoup4 requests pandas openpyxl nltk textstat -q

import pandas as pd
import requests
from bs4 import BeautifulSoup
import os
import re
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from google.colab import drive
import time
from urllib.parse import urlparse
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.2 MB/s eta 0:00:00


True

In [ ]:
# STEP 2: MOUNT GOOGLE DRIVE
# ============================================================================

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# STEP 3: CONFIGURATION - UPDATE THESE PATHS
# ============================================================================

BASE_PATH = '/content/drive/MyDrive/20211030 Test Assignment/20211030 Test Assignment/'
INPUT_FILE = BASE_PATH + 'Input.xlsx'
OUTPUT_FILE = BASE_PATH + 'output/Output_Data.xlsx'
EXTRACTED_ARTICLES_DIR = BASE_PATH + 'extracted_articles/'
POSITIVE_WORDS_FILE = BASE_PATH + 'MasterDictionary/positive-words.txt'
NEGATIVE_WORDS_FILE = BASE_PATH + 'MasterDictionary/negative-words.txt'
STOP_WORDS_DIR = BASE_PATH + 'StopWords/'

# Create output directories
os.makedirs(EXTRACTED_ARTICLES_DIR, exist_ok=True)
os.makedirs(BASE_PATH + 'output/', exist_ok=True)

print("✓ Configuration completed")
print(f"✓ Output will be saved to: {OUTPUT_FILE}")


✓ Configuration completed
✓ Output will be saved to: /content/drive/MyDrive/20211030 Test Assignment/20211030 Test Assignment/output/Output_Data.xlsx


In [ ]:
# STEP 4: LOAD DICTIONARIES AND STOP WORDS
# ============================================================================

def load_words_from_file(filepath):
    """Load words from a text file into a set."""
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            words = set(word.strip().lower() for word in f.readlines() if word.strip())
        return words
    except Exception as e:
        print(f"Warning: Could not load {filepath}: {e}")
        return set()

# Load positive and negative words
print("\n📖 Loading Master Dictionaries...")
positive_words = load_words_from_file(POSITIVE_WORDS_FILE)
negative_words = load_words_from_file(NEGATIVE_WORDS_FILE)
print(f"✓ Loaded {len(positive_words)} positive words")
print(f"✓ Loaded {len(negative_words)} negative words")

# Load stop words from all files
print("\n📖 Loading Stop Words...")
stop_words = set()
for filename in os.listdir(STOP_WORDS_DIR):
    if filename.endswith('.txt'):
        filepath = os.path.join(STOP_WORDS_DIR, filename)
        stop_words.update(load_words_from_file(filepath))
print(f"✓ Loaded {len(stop_words)} stop words")



📖 Loading Master Dictionaries...
✓ Loaded 2006 positive words
✓ Loaded 4783 negative words

📖 Loading Stop Words...
✓ Loaded 12768 stop words


In [ ]:
# STEP 5: WEB SCRAPING FUNCTIONS
# ============================================================================

def extract_article_text(url, url_id):
    """
    Extract article title and text from the given URL.
    Returns tuple: (title, article_text, success)
    """
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }

        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        # Remove unwanted elements
        for element in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            element.decompose()

        # Try to find title
        title = ""
        if soup.find('h1'):
            title = soup.find('h1').get_text().strip()
        elif soup.find('title'):
            title = soup.find('title').get_text().strip()

        # Try to find article content
        article_text = ""

        # Common article content selectors
        article_selectors = [
            'article',
            {'class': re.compile('entry-content|post-content|article-content|content')},
            {'id': re.compile('content|article|post')},
            'div.td-post-content',
            'div.tdb-block-inner'
        ]

        for selector in article_selectors:
            if isinstance(selector, str):
                content = soup.find(selector)
            else:
                content = soup.find('div', selector)

            if content:
                # Extract all paragraphs
                paragraphs = content.find_all('p')
                article_text = '\n'.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])
                if article_text:
                    break

        # Fallback: get all paragraphs
        if not article_text:
            paragraphs = soup.find_all('p')
            article_text = '\n'.join([p.get_text().strip() for p in paragraphs if p.get_text().strip()])

        # Combine title and article
        full_text = f"{title}\n\n{article_text}" if title else article_text

        return title, full_text, True

    except Exception as e:
        print(f"✗ Error extracting {url_id}: {str(e)}")
        return "", "", False

def save_article(url_id, text):
    """Save extracted article to a text file."""
    filepath = os.path.join(EXTRACTED_ARTICLES_DIR, f"{url_id}.txt")
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(text)

In [ ]:
# STEP 6: TEXT ANALYSIS FUNCTIONS
# ============================================================================

def clean_text(text, stop_words):
    """Remove stop words and clean text."""
    words = word_tokenize(text.lower())
    cleaned_words = [word for word in words if word.isalpha() and word not in stop_words]
    return cleaned_words

def calculate_sentiment_scores(words):
    """Calculate positive and negative scores."""
    positive_score = sum(1 for word in words if word in positive_words)
    negative_score = sum(1 for word in words if word in negative_words)

    return positive_score, negative_score

def calculate_polarity_score(positive_score, negative_score):
    """Calculate polarity score."""
    denominator = (positive_score + negative_score) + 0.000001
    polarity_score = (positive_score - negative_score) / denominator
    return polarity_score

def calculate_subjectivity_score(positive_score, negative_score, total_words):
    """Calculate subjectivity score."""
    denominator = total_words + 0.000001
    subjectivity_score = (positive_score + negative_score) / denominator
    return subjectivity_score

def count_syllables(word):
    """Count syllables in a word."""
    word = word.lower()
    vowels = 'aeiouy'
    syllable_count = 0
    previous_was_vowel = False

    for char in word:
        is_vowel = char in vowels
        if is_vowel and not previous_was_vowel:
            syllable_count += 1
        previous_was_vowel = is_vowel

    # Handle exceptions
    if word.endswith('e'):
        syllable_count -= 1
    if word.endswith('le') and len(word) > 2 and word[-3] not in vowels:
        syllable_count += 1
    if syllable_count == 0:
        syllable_count = 1

    return syllable_count

def is_complex_word(word):
    """Check if a word is complex (more than 2 syllables)."""
    # Exclude common exceptions
    exceptions = ['es', 'ed']
    if any(word.endswith(suffix) for suffix in exceptions):
        return False
    return count_syllables(word) > 2

def calculate_fog_index(avg_sentence_length, percentage_complex_words):
    """Calculate Fog Index."""
    fog_index = 0.4 * (avg_sentence_length + percentage_complex_words)
    return fog_index

def count_personal_pronouns(text):
    """Count personal pronouns (excluding 'US' as country)."""
    pronoun_pattern = r'\b(I|we|my|ours|us)\b'
    # Exclude 'US' when it appears to be referring to the country
    matches = re.findall(pronoun_pattern, text, re.IGNORECASE)

    # Filter out 'US' in uppercase (likely country reference)
    pronouns = [match for match in matches if match.upper() != 'US' or match != 'US']

    return len(pronouns)

def analyze_text(text):
    """
    Perform complete text analysis on the given text.
    Returns a dictionary with all computed variables.
    """
    # Tokenize sentences and words
    sentences = sent_tokenize(text)
    words = word_tokenize(text)

    # Clean text (remove stop words)
    cleaned_words = clean_text(text, stop_words)

    # 1. Sentiment Scores
    positive_score, negative_score = calculate_sentiment_scores(cleaned_words)

    # 2. Polarity Score
    polarity_score = calculate_polarity_score(positive_score, negative_score)

    # 3. Subjectivity Score
    total_words_after_cleaning = len(cleaned_words)
    subjectivity_score = calculate_subjectivity_score(
        positive_score, negative_score, total_words_after_cleaning
    )

    # 4. Average Sentence Length
    total_words = len([word for word in words if word.isalpha()])
    total_sentences = len(sentences) if len(sentences) > 0 else 1
    avg_sentence_length = total_words / total_sentences

    # 5. Complex Words
    complex_word_count = sum(1 for word in cleaned_words if is_complex_word(word))

    # 6. Percentage of Complex Words
    percentage_complex_words = (complex_word_count / total_words_after_cleaning * 100) if total_words_after_cleaning > 0 else 0

    # 7. Fog Index
    fog_index = calculate_fog_index(avg_sentence_length, percentage_complex_words)

    # 8. Word Count (cleaned)
    word_count = total_words_after_cleaning

    # 9. Syllables per Word
    total_syllables = sum(count_syllables(word) for word in cleaned_words)
    syllables_per_word = total_syllables / word_count if word_count > 0 else 0

    # 10. Personal Pronouns
    personal_pronouns = count_personal_pronouns(text)

    # 11. Average Word Length
    total_characters = sum(len(word) for word in cleaned_words)
    avg_word_length = total_characters / word_count if word_count > 0 else 0

    return {
        'POSITIVE SCORE': positive_score,
        'NEGATIVE SCORE': negative_score,
        'POLARITY SCORE': round(polarity_score, 4),
        'SUBJECTIVITY SCORE': round(subjectivity_score, 4),
        'AVG SENTENCE LENGTH': round(avg_sentence_length, 2),
        'PERCENTAGE OF COMPLEX WORDS': round(percentage_complex_words, 2),
        'FOG INDEX': round(fog_index, 2),
        'AVG NUMBER OF WORDS PER SENTENCE': round(avg_sentence_length, 2),
        'COMPLEX WORD COUNT': complex_word_count,
        'WORD COUNT': word_count,
        'SYLLABLE PER WORD': round(syllables_per_word, 2),
        'PERSONAL PRONOUNS': personal_pronouns,
        'AVG WORD LENGTH': round(avg_word_length, 2)
    }


In [ ]:
# STEP 7: MAIN PROCESSING
# ============================================================================

def main():
    """Main function to orchestrate the entire process."""

    print("\n" + "="*70)
    print("STARTING BLACKCOFFER TEXT ANALYSIS")
    print("="*70)

    # Load input data
    print("\n📂 Loading input data...")
    df = pd.read_excel(INPUT_FILE)
    print(f"✓ Loaded {len(df)} URLs to process")

    # Initialize results list
    results = []

    # Process each URL
    print("\n🌐 Starting web scraping and analysis...")
    print("-" * 70)

    for idx, row in df.iterrows():
        url_id = str(row['URL_ID'])
        url = row['URL']

        print(f"\n[{idx+1}/{len(df)}] Processing {url_id}...")
        print(f"URL: {url}")

        # Extract article
        title, article_text, success = extract_article_text(url, url_id)

        if success and article_text:
            # Save article
            save_article(url_id, article_text)
            print(f"✓ Article saved ({len(article_text)} characters)")

            # Analyze text
            analysis_results = analyze_text(article_text)

            # Combine with input data
            result = {
                'URL_ID': url_id,
                'URL': url,
                **analysis_results
            }
            results.append(result)
            print("✓ Analysis completed")
        else:
            print("✗ Failed to extract article")
            # Add empty results
            result = {
                'URL_ID': url_id,
                'URL': url,
                'POSITIVE SCORE': 0,
                'NEGATIVE SCORE': 0,
                'POLARITY SCORE': 0,
                'SUBJECTIVITY SCORE': 0,
                'AVG SENTENCE LENGTH': 0,
                'PERCENTAGE OF COMPLEX WORDS': 0,
                'FOG INDEX': 0,
                'AVG NUMBER OF WORDS PER SENTENCE': 0,
                'COMPLEX WORD COUNT': 0,
                'WORD COUNT': 0,
                'SYLLABLE PER WORD': 0,
                'PERSONAL PRONOUNS': 0,
                'AVG WORD LENGTH': 0
            }
            results.append(result)

        # Be polite to servers
        time.sleep(1)

    # Create output DataFrame
    print("\n" + "="*70)
    print("📊 Creating output file...")
    output_df = pd.DataFrame(results)

    # Save to Excel
    output_df.to_excel(OUTPUT_FILE, index=False)
    print(f"✓ Output saved to: {OUTPUT_FILE}")

    # Display summary
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE - SUMMARY")
    print("="*70)
    print(f"Total URLs processed: {len(df)}")
    print(f"Successfully analyzed: {len([r for r in results if r['WORD COUNT'] > 0])}")
    print(f"Failed extractions: {len([r for r in results if r['WORD COUNT'] == 0])}")
    print(f"\nExtracted articles location: {EXTRACTED_ARTICLES_DIR}")
    print(f"Output file location: {OUTPUT_FILE}")
    print("\n✅ ALL DONE!")

    return output_df


In [ ]:
# STEP 8: RUN THE ANALYSIS
# ============================================================================

if __name__ == "__main__":
    # Download the missing NLTK resource
    import nltk
    nltk.download('punkt_tab', quiet=True)

    output_df = main()

    # Display first few rows
    print("\n📋 Sample Output (first 5 rows):")
    print(output_df.head())



STARTING BLACKCOFFER TEXT ANALYSIS

📂 Loading input data...
✓ Loaded 147 URLs to process

🌐 Starting web scraping and analysis...
----------------------------------------------------------------------

[1/147] Processing Netclan20241017...
URL: https://insights.blackcoffer.com/ai-and-ml-based-youtube-analytics-and-content-creation-tool-for-optimizing-subscriber-engagement-and-content-strategy/
✓ Article saved (1230 characters)
✓ Analysis completed

[2/147] Processing Netclan20241018...
URL: https://insights.blackcoffer.com/enhancing-front-end-features-and-functionality-for-improved-user-experience-and-dashboard-accuracy-in-partner-hospital-application/
✓ Article saved (5078 characters)
✓ Analysis completed

[3/147] Processing Netclan20241019...
URL: https://insights.blackcoffer.com/roas-dashboard-for-campaign-wise-google-ads-budget-tracking-using-google-ads-ap/
✓ Article saved (1527 characters)
✓ Analysis completed

[4/147] Processing Netclan20241020...
URL: https://insights.blackcoff